# ALS reconstruction and KRG-alsDenoise, v3.4

Reconstruction and self-supervised denoising of 180° ALS micro-CT scans.
The workflow creates full/even/odd reconstruction stacks, estimates the
center of rotation with TomoPy Vo, and manages reconstruction queues,
training, inference and loss-history plots.

The ALS kernel runs `krg_workflow`; the n2i subprocess runs PyTorch.
Paths and parameters are set in action cells using a `run` object.
Existing schema-4 plans, queues, YAML, logs and checkpoints remain supported.
Fresh 2.5D runs default to NAFNet; baseline 2.5D and 3D U-Nets remain available.

Setup instructions are included below. Training requires
`RUN_TRAINING=True` in cell 13.


## Method and input requirements

Noise2Inverse360 trains a denoising model without a clean reference image. For each slice, it compares two reconstructions made from different measured angles: even-indexed projections and odd-indexed projections. The model learns features that agree between the two views. A third reconstruction from all projections supplies the image that will be denoised after training. The two training views share flats and darks, so their noise is not perfectly independent; inspect the result for removed features and remaining artifacts.

This notebook performs reconstruction and denoising. The ALS `microct` code performs the reconstruction; `KRG-alsDenoise` supplies the Noise2Inverse model training and inference commands. Reconstruction settings, especially center of rotation, must match across the full, even, and odd stacks. The notebook saves one plan and applies it to all three.

| Term | Meaning here |
|---|---|
| Projection | A raw 2D X-ray image acquired at one rotation angle. |
| Flat / dark | Calibration images acquired without the sample / without X-ray illumination. |
| Sinogram row | One detector row followed through all angles; it reconstructs one CT slice. |
| Center of rotation | Detector-center offset in pixels: `native center - detector width / 2`. Zero is the midpoint; negative offsets are valid. A wrong value can produce doubled edges. |
| Full / even / odd stack | TIFF slices reconstructed from all / even-indexed / odd-indexed projections. |
| 2.5D model | A 2D network that reads several neighboring slices as channels. |
| 3D model | A 3D network that reads and predicts cubic patches of the volume. |
| HDF5 virtual view | A small HDF5 file pointing at selected projections in the original file; it does not duplicate the raw images. |

Required input: one 180° parallel-beam HDF5 scan containing `/exchange/data` in `[angle, row, column]` order, `/exchange/theta` in degrees, and `/exchange/data_white` and `/exchange/data_dark`. The ALS path also needs detector dimensions, pixel size, angle count, sample distance, energy, angular range, and other standard ALS metadata. Cell 3 checks image layout and ALS metadata; cell 4 rechecks metadata before reconstruction. A folder of only reconstructed TIFFs cannot supply the independent even/odd training views; obtain the raw projections or separately reconstructed views before using Noise2Inverse360.

The detector width must be even. Flats and darks must be nonempty `[frame, row, column]` arrays matching the projections. Angles must increase monotonically and span approximately 180°. Metadata values used below must be finite, nonempty 1D arrays, not HDF5 scalar datasets.

| Required ALS metadata path | Meaning used by this notebook |
|---|---|
| `/measurement/instrument/detector/dimension_x`, `dimension_y` | Column and row counts, matching `/exchange/data`. |
| `/measurement/instrument/detector/pixel_size` | First entry: detector pixel size in **mm**; must be positive. |
| `/process/acquisition/rotation/num_angles`, `range` | Projection count and angular span in **degrees**, consistent with the image data and theta. |
| `/measurement/instrument/camera_motor_stack/setup/camera_distance` | Optional for this no-phase workflow. If present as a finite second entry, it is reported in mm; otherwise distance is recorded as unknown. A private metadata adapter avoids upstream indexing entry 1 when it is absent. |
| `/measurement/instrument/monochromator/energy` | First entry in **eV**. Required by the reader even though this notebook does not perform phase retrieval. |

A finite negative energy value can pass the reader but is not a valid physical energy; it does not enable phase retrieval. Obtain missing or malformed metadata from the beamline rather than substituting guessed values. The pixel-size conversion used for reconstruction is mm → cm, giving attenuation values in **1/cm**.

## First-time NERSC setup

Prerequisites: a NERSC account, access to the raw scan and clone access to
`remyferu/KRG-alsDenoise`. The steps below cover a new installation.

Terminal commands use Bash on NERSC. Notebook cells run in a Python kernel,
which must be selected separately from the Jupyter compute allocation.

### 1. Environment roles

| Environment | Tasks | Main software | Selection |
|---|---|---|---|
| `als832` | Notebook cells, projection processing, COR and reconstruction | ALS `microct` source, TomoPy, ASTRA, dxchange, NumPy, widgets | Select **ALS CT + N2I notebook** as the Jupyter kernel |
| `n2i` | Denoising training and inference | This repository, PyTorch with CUDA | Notebook launches its absolute `denoise` executable as a subprocess |

Create both environments once. Keep the notebook in the ALS kernel during the
whole workflow. Training runs in an n2i subprocess.
The source checkouts contain code, while the Conda prefixes contain Python and
installed packages. Cloning a repository alone does not install its dependencies.

Reconstruction uses ALS/TomoPy/ASTRA. The training configuration action runs
`denoise prepare` to create a YAML file. Its printed Tomocupy examples belong
to the upstream workflow; cells 9–11 produce and check the paired stacks
through ALS instead.

An existing ALS-provided `microCT` environment can serve as
the kernel after verifying the imports below. The
[ALS project](https://github.com/als-computing/microct#readme) describes its
beamline-specific service. The private installation below is an alternative for
standard Perlmutter Jupyter sessions. Their filesystem mounts may differ.

### 2. Create directories and download the source (login-node terminal)

Log in to NERSC through SSH, or use a login-node Jupyter terminal for lightweight
installation work. Use a compute allocation for expensive compilation or data
processing. These paths match the notebook's defaults. The Conda module follows
the [NERSC custom-environment procedure](https://docs.nersc.gov/development/languages/python/nersc-python/#option-2-custom-conda-environment):

```bash
module load conda
CT_SOURCE_DIR="$HOME/microct-n2i-source"
CT_SOFTWARE_DIR="$HOME/microct-n2i-envs"
mkdir -p "$CT_SOURCE_DIR" "$CT_SOFTWARE_DIR"
cd "$CT_SOURCE_DIR"
git clone https://github.com/als-computing/microct.git microct
git -C "$CT_SOURCE_DIR/microct" checkout e4116984003294af8dabba9d32db3fccb5ef2253
git clone https://github.com/remyferu/KRG-alsDenoise.git KRG-alsDenoise
```

The private KRG repository requires GitHub HTTPS authentication or
a configured SSH key with `git@github.com:remyferu/KRG-alsDenoise.git` as the
clone URL. If a directory already exists, inspect it and reuse the checkout;
do not rerun `git clone` into it or replace it. The ALS commit above is the pinned backend for this notebook.

### 3. Install the ALS reconstruction environment (login-node terminal)

For a fresh private installation, create `als832` from the pinned ALS source's
environment file, then add the notebook's supporting packages:

```bash
module load conda
CT_SOURCE_DIR="$HOME/microct-n2i-source"
CT_SOFTWARE_DIR="$HOME/microct-n2i-envs"
conda env create --prefix "$CT_SOFTWARE_DIR/als832" \
  -f "$CT_SOURCE_DIR/microct/environment.yml"
conda activate "$CT_SOFTWARE_DIR/als832"
conda install -c conda-forge h5py tifffile pyyaml pillow numexpr ipywidgets ipykernel
python -c 'import tomopy, dxchange, astra, h5py, tifffile, yaml, PIL, ipywidgets; print("ALS imports OK")'
python -m pip install --no-deps -e "$CT_SOURCE_DIR/KRG-alsDenoise/workflow"
krg-workflow doctor
python -m ipykernel install --user --name als832-n2i \
  --display-name "ALS CT + N2I notebook"
```

The [pinned ALS environment](https://github.com/als-computing/microct/blob/e4116984003294af8dabba9d32db3fccb5ef2253/environment.yml)
uses Python 3.8 and older Matplotlib/Jupyter pins. This is a source-based setup
recipe rather than a solver-verified NERSC lock file. If Conda cannot solve it, keep an
existing working ALS environment and resolve the solver error before proceeding;
do not install TomoPy into n2i as a substitute. `ipywidgets` is needed by the ALS
backend itself. Zoom displays do not require an interactive Matplotlib backend.

In JupyterLab choose **Kernel → Change Kernel → ALS CT + N2I notebook**. Reload
JupyterLab if the kernel is missing. Changes to an already running kernel's
environment require a kernel restart. The first code cell prints `sys.executable`;
for this installation it should end in `microct-n2i-envs/als832/bin/python`.
See the [NERSC custom-kernel procedure](https://docs.nersc.gov/services/jupyter/how-to-guides/#how-to-use-a-conda-environment-as-a-python-kernel).

The `krg-ct-workflow` package lives in `workflow/` and requires no Torch. Its
scientific dependencies are supplied by the ALS environment, so use `--no-deps`
to preserve those compatible versions. If pip/setuptools is too old for editable
installation, install `setuptools>=61,<76` and `pip<25.1` inside this Python 3.8
environment, then retry. Restart the notebook kernel after installing/updating it.

### 4. Install the denoising environment (login-node terminal)

For a new n2i prefix, install a CUDA-enabled PyTorch build and this package with
its dependencies. The example uses an official PyTorch 2.7.1 / CUDA 11.8 wheel;
the notebook requires PyTorch >= 2.0. Use a site-compatible build when necessary.
[PyTorch wheel choices](https://pytorch.org/get-started/previous-versions/#v271)
and [NERSC guidance](https://docs.nersc.gov/machinelearning/pytorch/) explain the
custom-environment option.

```bash
module load conda
CT_SOURCE_DIR="$HOME/microct-n2i-source"
CT_SOFTWARE_DIR="$HOME/microct-n2i-envs"
conda create --prefix "$CT_SOFTWARE_DIR/n2i" python=3.11 pip
conda activate "$CT_SOFTWARE_DIR/n2i"
python -m pip install 'numpy<2.0'
python -m pip install torch==2.7.1 --index-url https://download.pytorch.org/whl/cu118
python -m pip install -e "$CT_SOURCE_DIR/KRG-alsDenoise"
python -m pip check
MPLBACKEND=Agg PYTHONNOUSERSITE=1 "$CT_SOFTWARE_DIR/n2i/bin/denoise" train -h
```

CLI help must include `--architecture` and `nafnet25d`. The adaptation is
included in this repository and has no BasicSR dependency. For an existing n2i prefix,
skip `conda create` and the dependency installations; activate that prefix and run
`python -m pip install --no-deps -e "$HOME/microct-n2i-source/KRG-alsDenoise"`,
then check imports and CLI help. `--no-deps` is only for an environment with the
dependencies already installed. The repository's original
`envs/legacy/denoise_environment.yml` is a machine-specific upstream export, not the
portable first-time recipe for this notebook.

### 5. Start Jupyter on a compute allocation

Open [NERSC JupyterHub](https://jupyter.nersc.gov/), start a **single-node GPU**
compute server and open
`microct-n2i-source/KRG-alsDenoise/notebooks/Noise2Inverse360_HPC_v3.4.ipynb`.
For the full-volume example, choose **Exclusive GPU Node**, or a Configurable job
with sufficient host RAM and time. NERSC documents four GPUs and full node memory
for the exclusive GPU configuration; a shared GPU server has one GPU and a
quarter of the node's host resources. [Allocation options](https://docs.nersc.gov/services/jupyter/reference/).

Select **ALS CT + N2I notebook** as the kernel. Activating Conda or starting an
allocation in another terminal does not relocate or change the running kernel.
The workflow uses one node: training uses all assigned GPUs, inference uses one.
In a terminal opened **inside this compute Jupyter session**, check:

```bash
hostname
echo "Slurm job: ${SLURM_JOB_ID:-unset}"
echo "GPU assignment: ${CUDA_VISIBLE_DEVICES:-unset}"
nvidia-smi
"$HOME/microct-n2i-envs/n2i/bin/python" -c 'import torch; print(torch.__version__, torch.version.cuda); assert torch.cuda.is_available(); print("Visible GPUs:", torch.cuda.device_count())'
```

This checks CUDA availability; it does not train a model. Keep the scheduler's
`CUDA_VISIBLE_DEVICES` assignment. If it is unset, resolve the allocation/kernel
setup before training instead of guessing identifiers from `nvidia-smi`. If either
environment needs site CUDA modules, apply them through a kernel/helper wrapper
following [NERSC's helper instructions](https://docs.nersc.gov/services/jupyter/how-to-guides/#how-to-customize-a-kernel-with-a-helper-shell-script).
CUDA modules must also be available to the subprocess environment.

### 6. Stage the scan and edit cell 2

The notebook does not download or copy scan data. Place the raw HDF5 scan in a
directory visible from the compute kernel. On standard Perlmutter, an example is
`$PSCRATCH/raw_microct/sample.h5`. Set the source filename and verify that
the file meets the projection/flat/dark/metadata requirements in the notebook’s Method and input requirements section.
Reconstructed TIFFs alone are insufficient for the notebook's projection splitting.

| Cell 2 input | Value |
|---|---|
| `INPUT_ROOT` | Folder containing raw `.h5`/`.hdf5` files; replace the example folder or use an explicit absolute `Path` |
| `SCAN_PATH` | Absolute raw scan path; confirm it in cell 3 |
| `RUN_LABEL` | A name for this scan's COR, settings and model combination; use a new label when those change |
| `ALS_MICROCT_DIR` | Checkout containing `backend/ALS_recon_helper.py`, normally `Path.home() / 'microct-n2i-source' / 'microct'` |
| `DENOISE_CMD` | `[str(Path.home() / 'microct-n2i-envs' / 'n2i' / 'bin' / 'denoise')]` |
| `OUTPUT_ROOT_OVERRIDE` | Leave `None` to use `$PSCRATCH`/`$SCRATCH`, or enter an absolute output scratch directory |

In Python, `Path.home()` expands the account home directory; `Path('$HOME/...')` does not
expand Bash variables. `/alsdata` and `/alsuser/pscratch` are service-specific
paths, not assumed mounts on standard Perlmutter. Stage to a stable input path:
moving the raw file later breaks the saved plan and virtual projection views.

### 7. Directory layout

Keep code, environments and notebooks in home (subject to quota); use scratch for
large input copies, TIFFs and active model outputs. Standard output layout:

```text
$HOME/microct-n2i-source/microct/backend/ALS_recon_helper.py
$HOME/microct-n2i-source/KRG-alsDenoise/notebooks/Noise2Inverse360_HPC_v3.4.ipynb
$HOME/microct-n2i-envs/als832/         # notebook Python
$HOME/microct-n2i-envs/n2i/           # denoising Python
$PSCRATCH/noise2inverse360/<scan>/<RUN_LABEL>/
  run_plan.json, scan_metadata.json
  projection_views/even.h5, odd.h5
  <scan>_rec/, <scan>_rec_0/, <scan>_rec_1/
  <scan>_rec_config.yaml
  TrainOutput/results/reference_metrics.csv
  TrainOutput/{resume,best_val_model,best_lcl_model,best_edge_model}.pth
  <scan>_denoised_slices_nafnet25d/
  <scan>_denoised_volume_2.5d_nafnet25d/
  logs/, training_history/
$PSCRATCH/noise2inverse360/reconstruction_queue_v3.json
```

The selected output root can differ from this example. Full/even/odd float32
stacks for 624 × 2560 × 2560 occupy about 46 GiB on disk. Training with four
processes holds about 122 GiB of paired input arrays in host RAM before overhead.
GPU VRAM is separate from host RAM. Read the volume section for its additional
memory needs. Scratch is working storage; preserve accepted data and small run
records in durable storage before cleanup.

### 8. Update an existing installation

After stopping jobs that use this checkout, update it in a terminal:

```bash
cd "$HOME/microct-n2i-source/KRG-alsDenoise"
git status --short
git pull --ff-only
```

For the first upgrade from v3.3, also activate the ALS environment and install
`python -m pip install --no-deps -e "$HOME/microct-n2i-source/KRG-alsDenoise/workflow"`.
Open the new v3.4 notebook; the v3.3 notebook remains available.

Preserve local notebook edits before pulling. Editable installation reads code
from this checkout, so routine code updates need no package reinstall; changed
dependencies or entry points do. Reopen the updated notebook, restart its kernel and rerun initialization.
Changing ALS backend files invalidates saved source identity, so keep the
pinned checkout unchanged during active runs.

### Setup troubleshooting

| Symptom | Next step |
|---|---|
| `ModuleNotFoundError: tomopy`, `astra` or `dxchange` in a cell | Check the printed kernel Python; select the ALS kernel and verify ALS imports |
| `ModuleNotFoundError: torch` from `denoise` | Check `DENOISE_CMD` and the n2i environment; PyTorch belongs there |
| CLI help lacks `--architecture` | Update/install KRG-alsDenoise in n2i; an older Noise2Inverse executable is being used |
| Invalid `matplotlib_inline` backend in the CLI | Confirm the subprocess uses `MPLBACKEND=Agg`; the notebook's launcher sets it |
| NumExpr thread-count error | Restart the kernel and run cell 1 first. If NumExpr is preloaded, set `NUMEXPR_MAX_THREADS` in the kernel launcher to at least the host's detected logical CPU count before Python starts |
| Kernel remains missing | Reload JupyterLab after registration; check the kernelspec points to an existing ALS Python |
| No accessible scratch or scan folder | Set cell 2's input and scratch paths to locations visible on the actual compute node |

If NumExpr loads before cell 1, inspect the host CPU count with
`getconf _NPROCESSORS_ONLN` in the compute-session terminal. In JupyterLab's text
editor open `$HOME/.local/share/jupyter/kernels/als832-n2i/kernel.json`. Add an
`env` field at the same level as `argv`, using this object as its value. Keep
the existing interpreter path:

```json
{
  "NUMEXPR_MAX_THREADS": "256",
  "NUMEXPR_NUM_THREADS": "8"
}
```

`256` is an example: use a maximum at least as large as that host's detected
logical CPU count. Keep valid JSON commas between the existing fields, then
restart the kernel. Cell 1 reduces the initial active count to the available CPU
limits. [NERSC kernel environment instructions](https://docs.nersc.gov/services/jupyter/how-to-guides/#how-to-set-environment-variables-for-a-python-kernel).

Initialize the notebook with cells 1–4, then follow Workflow and restart.
Training requires `RUN_TRAINING=True` in cell 13. A fresh NERSC installation
and reconstruction have not been verified for this setup. See
[denoising details](../docs/NAFNET_CT.md) for architecture and loss settings.


## Workflow and restart

Run cells individually, from the top. Avoid **Run All**, which also executes
reconstruction and inference. Optional cells are labeled in their titles.

| Cells | Purpose |
|---|---|
| 1–4 | Initialize, select paths, inspect scan, configure reconstruction |
| 6, 9–11 | Estimate/freeze COR, enqueue, reconstruct, inspect paired stacks |
| 5, 7–8 | Optional projection alignment and manual COR review |
| 12–13 | Configure training and explicitly enable fresh training/resume |
| 14 | Optional training-history plot, including logs from other runs |
| 15–16 | Inspect denoising and apply it to the full volume |
| 17–20 | Optional queue repair and previewed scratch cleanup |

After a kernel restart, rerun 1–4 for the **same scan and run label**. Cell 6
restores an existing frozen plan; queue reconstruction skips validated completed
stages. Run 11–12 to reconnect existing stacks and saved training configuration.
Cell 13 resumes only if `TrainOutput/resume.pth` exists and `RUN_TRAINING=True`.
Changing reconstruction identity requires a new run label. Updating installed
helper code requires a kernel restart; cell 1 reports its version and source path.

Viewers hold their run context. Selecting a different scan/run or changing
reconstruction settings invalidates old controls; recreate the viewer. Keep
configuration edits together and rerun their configuration cell before continuing.


In [1]:
# 1. Initialize workflow and check the kernel
from pathlib import Path
import json, os, sys
import krg_workflow
from krg_workflow.runtime import configure_reconstruction_threads, available_cores_minus
THREADS = configure_reconstruction_threads()  # Before scientific imports
from krg_workflow import api as ct, presets, queue, cleanup
from krg_workflow.history import plot_training_history
from krg_workflow.notebook_ui import live_log_box, manual_review
try:
    run.invalidate()
except (NameError, AttributeError):
    pass
run = None
print('Workflow:', krg_workflow.__version__, '| source:', krg_workflow.__file__)
print('Kernel:', sys.executable, '| threads:', THREADS)
print('Slurm:', os.environ.get('SLURM_JOB_ID'), '| GPUs:', os.environ.get('CUDA_VISIBLE_DEVICES'))


Workflow: 3.4.0 | source: /global/u1/r/remyferu/microct-n2i-source/KRG-alsDenoise/workflow/src/krg_workflow/__init__.py
Kernel: /global/homes/r/remyferu/microct-n2i-envs/als832/bin/python | threads: {'detected_cpus': 256, 'maximum': 256, 'active': 8}
Slurm: None | GPUs: None


## Paths and scan selection: cells 2–3

Set `INPUT_ROOT` and `SCAN_PATH` to a raw scan visible from the compute node.
The example path is `$PSCRATCH/raw_microct/sample.h5`. Cell 2 lists `.h5` files
when the input directory is accessible. `RUN_LABEL` identifies one COR,
reconstruction settings and model configuration; changed reconstruction identity
or a new model comparison requires a new label.

`ALS_MICROCT_DIR` selects the ALS checkout; `DENOISE_CMD` selects the n2i
executable. `OUTPUT_ROOT_OVERRIDE=None` uses NERSC scratch. An absolute `Path`
overrides that location. Cell 3 checks the scan and creates its run record.
Rerunning selection invalidates previous run objects and viewers.


In [4]:
# 2. Select workspace and scan paths
INPUT_ROOT = Path(os.environ.get('PSCRATCH') or os.environ.get('SCRATCH') or '/alsdata')/'ALS-12649_koo-scratch'
SCAN_PATH = INPUT_ROOT/'20260605_162159_Hex20C1.h5'  # Raw HDF5 scan
RUN_LABEL = 'v3_4_nafnet_1'         # New label for changed source/COR/settings
OUTPUT_ROOT_OVERRIDE = None       # None uses NERSC scratch
ALS_MICROCT_DIR = Path.home()/'microct-n2i-source/microct'
DENOISE_CMD = [str(Path.home()/'microct-n2i-envs/n2i/bin/denoise')]
if run is not None:
    run.invalidate()  # Old viewers must not operate on a new selection
workspace = ct.Workspace.create(ALS_MICROCT_DIR, DENOISE_CMD, OUTPUT_ROOT_OVERRIDE)
print('Selected scan:', SCAN_PATH, '| output root:', workspace.output_root)
if INPUT_ROOT.is_dir():
    print('Available scans:', [p.name for p in sorted(INPUT_ROOT.glob('*.h5'))])


Selected scan: /pscratch/sd/r/remyferu/ALS-12649_koo-scratch/20260605_162159_Hex20C1.h5 | output root: /pscratch/sd/r/remyferu/noise2inverse360
Available scans: ['20260605_162159_Hex20C1.h5', '20260605_165346_Hex800C1.h5', '20260605_172716_Hex1000C1.h5', '20260605_174737_Hex1250C1.h5']


In [5]:
# 3. Inspect the selected scan
run = ct.inspect_run(workspace, SCAN_PATH, RUN_LABEL, previous=run)
print('Run directory:', run.run_dir)


Camera distance has no usable second entry; recorded as unknown. This workflow does not use phase retrieval or camera distance.
{
  "shape": [
    1313,
    624,
    2560
  ],
  "dtype": "uint16",
  "flats": 15,
  "darks": 10,
  "angle_start_deg": 0.0,
  "angle_end_deg": 180.0,
  "angle_step_deg": 0.13719511032104492,
  "pixel_um": 3.243,
  "distance_mm": null,
  "stored_energy_ev": -1678684.596649,
  "energy_kev": null
}
Full/even/odd TIFFs: 45.70 GiB; paired RAM per training process: 30.47 GiB.
Run directory: /pscratch/sd/r/remyferu/noise2inverse360/20260605_162159_Hex20C1/v3_4_nafnet_1


## Configure reconstruction: cell 4

The notebook calls the ALS backend's `helper.reconstruct` function using the settings below; the upstream ALS notebook is not required. `method='fbp'` selects ASTRA FBP (CPU or GPU); `gridrec` selects CPU TomoPy gridrec. The explicit method keeps queued runs consistent across machines. `fc` is the backend's low-pass cutoff; start at 0.75 and compare previews. Reconstruction uses full detector resolution, circular masking, and upstream attenuation scaling.

`preprocessing` operates **after flat/dark normalization, before minus-log**: the outlier threshold is in transmission units, not the legacy raw-count threshold of 750. Zero disables outlier removal. `sm_size=0` disables the upstream SAREPY-style stripe routine; enabling it also requires `snr` and `la_size`. `postprocessing.ringSigma=0` disables Fourier-wavelet stripe removal; a positive value enables it with `ringLevel`. Phase retrieval and exposure-ratio correction are not exposed by this backend wrapper; this notebook assumes equal flat/projection exposure.

Start with corrections off, then compare their effect on previews. Use the same settings for full/even/odd. `als_chunk_sino` limits rows loaded per call; 25 is a conservative starting point. No projection downsampling or 360-to-180 conversion is applied.

| Setting | Interpretation |
|---|---|
| `method` | `fbp`: ASTRA FBP; `gridrec`: TomoPy gridrec. These can produce different filtering and image appearance. |
| `fc` | Cutoff in `(0, 1]`; smaller values smooth more. FBP and gridrec implement filtering differently, so the same number does not imply identical results. |
| `preprocessing.minimum_transmission` | Positive floor on normalized transmission before minus-log; prevents taking log of zero/negative values. |
| `preprocessing.outlier_diff_1D`, `outlier_size_1D` | Transmission difference threshold and window size for filtering along the angle dimension; threshold 0 disables it. |
| `preprocessing.sm_size`, `la_size`, `snr` | Small/large stripe windows and sensitivity for upstream stripe removal; `sm_size=0` disables it. Use suitable odd window sizes when enabled. |
| `postprocessing.ringSigma`, `ringLevel` | Fourier-wavelet damping and decomposition level; `ringSigma=0` disables it. |
| `als_chunk_sino` | Number of detector rows loaded per reconstruction call. It does not limit Noise2Inverse's memory use. |

Inspect changes before freezing the plan. Even a chunk-size or CPU/GPU change changes this notebook's saved settings and requires a new run label. Full/even/odd must be regenerated together when image-forming settings change.

In [6]:
# 4. Configure reconstruction
RECON_SETTINGS = presets.reconstruction_settings()
# Example: RECON_SETTINGS['als_chunk_sino'] = 25
ALS_USE_GPU = None  # None detects an allocated GPU; False uses CPU
ct.configure_reconstruction(run, workspace, RECON_SETTINGS, use_gpu=ALS_USE_GPU)


RuntimeError: This appears to be a NERSC login node. Start a compute-node Jupyter server before reconstruction or denoising.

## Image controls

Zoom enlarges a region; Center x, Center y and the arrow buttons move the crop.
Reset view restores the full image. Comparison panels share pixel coordinates,
with crop bounds displayed above the images.

Black and White set display contrast. Reconstruction panels share one window;
the residual has a separate window selected through the Contrast menu. Windows
remain fixed across crops and slices. Reset contrast restores the initial window.

Crops retain native pixels and use nearest-neighbor magnification. TIFF browsers
load one slice at a time. Display settings do not alter saved images or invoke
reconstruction or inference.

Controls require `ipywidgets`; otherwise the viewer displays a static image.
For a static crop, pass `view_options={'zoom': 8, 'center': (x, y)}` to
`ct.inspect_pairs`, `ct.inspect_alignment`, `ct.inspect_denoising` or
`ct.denoise_volume`. Coordinates are original pixel indices. An interactive
Matplotlib backend is not required.


## Projection alignment check (optional)

The next cell estimates alignment by phase correlation and displays a
first/opposite projection overlay. This provides a visual check independent
of Vo centering. It does not set Vo’s starting point or modify a saved plan.


In [ ]:
# 5. Check projection alignment (optional)
BRIGHT_RATIO = 1.0
alignment_viewer = ct.inspect_alignment(run, bright_ratio=BRIGHT_RATIO)


## Vo centering and run plan

The next cell calls `tomopy.find_center_vo()` on three detector-row sinograms
prepared with the reconstruction settings: flat/dark normalization, transmission
floor, minus-log and selected corrections. Agreement between estimates permits
saving their median midpoint offset as COR. No candidate reconstructions are scored.

Default rows are at 25%, 50% and 75% of detector height. Select rows containing
sample material. Flat or nonfinite rows, estimates near native Vo bounds, or a
spread greater than 1 pixel require review. This spread threshold is provisional;
it is not an accuracy calibration. Vo requires approximately uniform 180° angles.

`VO_OPTIONS` overrides parameters returned by `vo_options()`. The default fine
step is 0.1 pixels and `ncore=1` limits native parallelism on supported TomoPy
versions. The step sets numerical resolution, not measured accuracy. Vo uses
its own Fourier coarse/fine search. Results in `auto_cor_vo` record the TomoPy
version, native centers and midpoint offsets.

For REVIEW REQUIRED, revise the rows or native bounds, or review and apply a
manual offset. An existing frozen plan restores its saved COR. Re-estimation
of a frozen run requires a new run label. After saving a plan, enqueue and
reconstruct in cells 9–10.


In [ ]:
# 6. Estimate and save COR with TomoPy Vo
AUTO_COR_ROWS = None  # None uses 25%, 50%, 75% of detector height
VO_OPTIONS = {'ncore': available_cores_minus(4)}       # Default ncore=1, step=0.1
# To use allocated CPU cores minus four: VO_OPTIONS = {'ncore': available_cores_minus(4)}
plan, cor_result = ct.estimate_cor(run, rows=AUTO_COR_ROWS, options=VO_OPTIONS)


In [ ]:
# 7. Review COR manually (optional)
RUN_MANUAL_REFINEMENT = False
MANUAL_ROWS = None  # None selects the middle detector row
if RUN_MANUAL_REFINEMENT:
    manual_controls = manual_review(run, rows=MANUAL_ROWS, adjustment=0., apply=False)
else:
    print('Manual review disabled; continue with the saved automatic plan.')


## Optional: save a reviewed COR

Skip this cell when the automatic COR cell has already saved the plan. After manual review,
click **Apply selected COR**, then run the next cell. Alternatively, enter a numeric
midpoint offset in `COR_OVERRIDE` (zero and negative values are valid).

The cell also restores an existing plan after a restart. Frozen reconstruction settings
and COR cannot be changed under the same run label; use a new label for a different COR.
Moving a slider alone does not apply a selection.

In [ ]:
# 8. Save a reviewed COR (optional)
COR_OVERRIDE = None  # Numeric midpoint offset in detector pixels
plan = ct.freeze_selected_cor(run, override=COR_OVERRIDE)


## Queue and reconstruct: cells 9–10

Cell 9 adds the selected `run_plan.json` to
`workspace.output_root/reconstruction_queue_v3.json`. Run it only after COR has produced a
saved plan. Cell 10 prepares even/odd projection views and reconstructs queued
runs **sequentially**; adding an entry alone does not start reconstruction.

The queue can contain several scans/run labels. It records a stable `id` and a
`plan_path` for each entry. Preflight checks each plan and its source/settings;
missing or mismatched entries are blocked, duplicates skipped, and ordinary errors
recorded while later valid runs continue. An interrupt stops further entries.
The queue report is `workspace.output_root/queue_report_v3.json`; each run also has logs and
completion records. On a retry, completed stacks are skipped only after their
TIFFs pass validation; an incomplete tracked stage is reconstructed again.

Cell 11 inspects the run selected in cell 2, not every run that happened to be in
the queue. To remove an entry or repair its path, use cells 17–18. Removing a queue
entry does not delete reconstructed images. The queue lock is
`workspace.output_root/reconstruction_queue_v3.json.lock`; run actions can also use a
`.run.lock` inside their run directory. A lock file marks an active operation;
confirm the owner has stopped before considering stale-lock repair.


In [ ]:
# 9. Add the run to the reconstruction queue
run.assert_current(run.snapshot(), check_source=True)
queued = queue.enqueue_plan(workspace.queue_path, run.plan_path)
print('Queued entries:', len(queued['entries']))


In [ ]:
# 10. Reconstruct queued runs
from krg_workflow.runtime import require_compute_session
require_compute_session()
run.assert_current(run.snapshot(), check_source=True)
report = queue.execute_queue(workspace.queue_path, run.backend, run.settings)
print('Queue report:', report)


## Check the three reconstruction stacks: cell 11

The same row should show the same structures in all three images. Even and odd reconstructions will be noisier and can have more angular undersampling artifacts. Their measured projection indices are disjoint, although shared flat/dark calibration and detector artifacts can leave correlated errors; denoising is not a substitute for this check. Preserve float32 intensity values. The viewer uses the full reconstruction's contrast range for all three images.

Check more than the middle slice, including regions near the top and bottom. Stop before training if boundaries are displaced, the stacks have different orientation, or one view shows a gross intensity mismatch. All output filenames represent the same zero-based detector row: `recon_00000.tiff` is row 0. Shared contrast helps reveal differences but is not a quantitative test of independence or reconstruction accuracy.

In [ ]:
# 11. Inspect reconstruction stacks
pair_viewer = ct.inspect_pairs(run)


## Configure and train

Fresh 2.5D runs default to `nafnet25d`; set `TRAIN_ARCHITECTURE = 'unet'` in
cell 12 for the baseline. **2.5D** uses neighboring axial slices as input and
predicts their center slice. **3D** uses cubic patches and predicts a volume patch;
it uses the 3D U-Net and generally needs more computation/memory. Cell 12 selects
the mode/architecture and applies its preset. Compare five slices first,
then set `TRAIN_OVERRIDES = {'n_slices': 7}` or 9 in a fresh run. BF16 is optional; set `TRAIN_OVERRIDES = {'precision': 'fp32'}` for a precision comparison. These are starting parameters, not tuned optima.

Cell 13 prints a disabled message until `RUN_TRAINING = True`.
It then uses all allocated GPUs, retains the live log box, and saves automatic
history plots even on interruption. Early stopping uses the trailing mean of
evaluation total loss, with its own window and minimum improvement threshold.
History display smoothing is configured separately.

| Setting in cell 12's preset/overrides | Meaning |
|---|---|
| `psz` / `n_slices` | 2.5D square patch width and number of neighboring slices; use an odd slice count |
| `mbsz` | Batch size per GPU; four GPUs with batch 4 process a total batch of 16 |
| `precision` | `bf16` for supported GPUs such as A100; `fp32` for a precision comparison |
| `psz_3d` / `nb_patches_3d` | 3D cube width and number of sampled training patches per epoch |
| `lr` | Adam learning rate; YAML prepared for a new run starts at 0.001 |
| `warmup` | Optimizer updates before adding weighted LCL; initial YAML uses 2000 |
| `maxep` / `patience` | Maximum epochs / eligible epochs without qualifying mean-total-loss improvement; defaults 2000 / 100 |
| `early_stopping_window` | Number of post-warmup epochs in the trailing total-loss mean; default 10 |
| `min_delta` | Required absolute decrease from the best qualifying total-loss mean; default 0.001 |

An **epoch** visits the sampled training patches and then runs an evaluation pass.
Evaluation currently samples the training dataset again; it is not a separate
held-out accuracy estimate. L1 measures paired absolute error. LCL is an output
contrast regularizer weighted by 0.01 after warmup. L2/MSE and SSIM loss are
recorded only for reference. Optimization and evaluation share the total-loss
assembly; stopping monitors that total rather than an individual reporting metric.
No learning-rate scheduler is enabled.

Warmup and transition epochs do not enter the stopping window. The first complete
window establishes its baseline; later decreases must exceed `min_delta` to reset
patience. Raw-score checkpoints are selected independently. Resume restores the
window and counter when policy matches. Older checkpoints, or changed window/threshold
settings, start a fresh post-warmup window. `patience=0` disables early stopping.

Saved training settings and normalization are kept when `TrainOutput` exists;
editing a fresh-run preset does not alter an existing model. Cell 13 automatically
adds `--resume` when a valid `resume.pth` exists, otherwise starts fresh. A U-Net
checkpoint cannot initialize NAFNet. Run cell 14 on a saved log while
leaving training disabled.


In [ ]:
# 12. Configure training
TRAIN_MODE = '2.5d'             # '2.5d' or '3d'
TRAIN_ARCHITECTURE = 'nafnet25d'  # 'unet' also available; 3D uses unet3d
TRAIN_OVERRIDES = {'early_stopping_window': 10, 'min_delta': 0.001}
training = ct.configure_training(run, workspace, TRAIN_MODE, TRAIN_ARCHITECTURE,
                                 overrides=TRAIN_OVERRIDES, on_line=live_log_box())


In [ ]:
# 13. Train or resume the denoising model
RUN_TRAINING = False  # Enable training explicitly
ct.train_run(run, workspace, enabled=RUN_TRAINING, on_line=live_log_box())


## Training history and logs

Training saves epoch-average L1/LCL curves, a trailing 10-epoch moving average,
warmup markers and checkpoint annotations under `training_history` as PNG, SVG
and CSV. When present, MSE and SSIM loss have additional panels. Complete epochs
can also be plotted after interruption or failure.

Cell 14 reads the current log once. Rerun it to refresh a growing log, or set
`HISTORY_LOG` to another run’s log. Plotting requires neither a checkpoint nor
PyTorch in the ALS kernel. Duplicate resumed epochs use the latest complete record.

`L1 loss: mean, first => last` reports the average, first and last batch values.
`rate` measures seconds for the training/evaluation epoch. L1 and weighted LCL
logs sum the two paired directions; MSE and SSIM loss average them. Summaries
use the final GPU rank’s batches, rather than an aggregate across ranks.

SSIM loss is `1 - SSIM`, using the fixed normalized intensity range in
`train.ssim_data_range`. If absent, training derives the range once from both
stacks and saves it in YAML. Comparisons require consistent range and window
settings. In 3D, SSIM and LCL use the center axial plane; MSE uses the full patch.
These reference metrics do not measure clean-reference accuracy or affect
optimization, checkpoints or patience. Total loss is logged and plotted separately;
the dashed stopping curve uses the actual logged monitor values. Reference measurements are also stored
in `TrainOutput/results/reference_metrics.csv` and the resume checkpoint.

The live log box displays a scrolling tail with ANSI escapes removed. Complete
logs retain the original output under `logs/`. Startup lines can appear once
per GPU process.


In [ ]:
# 14. Show training history (optional)
HISTORY_LOG = None  # Or Path('/absolute/path/to/train.log')
MOVING_AVERAGE_EPOCHS = 10  # Plot smoothing; does not change early stopping
if HISTORY_LOG is None:
    run.assert_current(run.snapshot())
    history_log, history_output = run.run_dir/'logs/train.log', run.run_dir/'training_history'
else:
    history_log = Path(HISTORY_LOG).expanduser().resolve()
    history_output = history_log.parent/'training_history'
history = plot_training_history(history_log, history_output, MOVING_AVERAGE_EPOCHS)


## Slice inspection: cell 15

2.5D produces a single-slice preview. In 3D, the slice command is unavailable;
cell 15 displays rows from an existing denoised volume. Inspect training previews
in `TrainOutput/results` before the first 3D volume run.

| `CHECKPOINT` | Selection criterion |
|---|---|
| `lcl` | Lowest recorded evaluation LCL loss |
| `val` | Lowest recorded evaluation L1 loss |
| `edge` | Highest recorded edge score |

Compare original, denoised and residual images across several rows at matched
zoom and contrast. Check pores, interfaces and fine structure. Features visible
in the residual can indicate removed signal; a higher edge score can retain
noise as well as detail. Checkpoint scores are internal selection criteria.

`TEST_SLICE` is a zero-based row index in the full TIFF stack. Repeating a slice
command replaces that row’s output, including after a checkpoint change. Preserve
comparison outputs under distinct names. Cell 16 uses its own `VOLUME_CHECKPOINT`
and the normalization saved in YAML; changing the setting does not regenerate
an existing volume.


In [ ]:
# 15. Inspect denoising on one slice
CHECKPOINT = 'lcl'  # Compare 'val' or 'edge' where available
TEST_SLICE = run.shape[1]//2
denoising_viewer = ct.inspect_denoising(run, workspace, row=TEST_SLICE,
                                      checkpoint=CHECKPOINT, on_line=live_log_box())


## Denoise the full volume: cell 16

For 2.5D, review several test slices in cell 15 first. For 3D, review the training
preview images, run volume inference, then use cell 15 to inspect the saved rows.
Select an available checkpoint before starting. Training can use several GPUs;
this volume command uses **one GPU**, plus substantial **host RAM**.

This repository's 2.5D profiler uses the actual input channel count; models with
5, 7 or 9 slices are supported. Its prediction buffer is **float32**. For a
624 × 2560 × 2560 volume with 256-pixel patches and 50% overlap, that buffer alone
is about **55 GiB**: `624 × 19 × 19 × 256 × 256 × 4 bytes`. Input, stitching,
temporary arrays and the notebook process require additional RAM. This is a
planning estimate, not a measured peak, and GPU VRAM cannot replace host RAM.
3D stitches predictions incrementally but retains the full input and accumulation
arrays; it still needs substantial host RAM.

Neither the notebook nor this backend deletes an existing volume directory to
restart. Cell 16 refuses to overwrite it. There is **no volume resume**: preserve
partial output and use a new output directory/run for a retry. The CLI's
`--output-dir` can select a separate destination outside this cell, but assembling
independently inferred subvolumes is not automated here. If full 2.5D inference
does not fit, its single-slice command remains available.

NAFNet outputs end in `_nafnet25d`, so they are distinguishable from baseline
outputs. After inference, the notebook checks filenames, dimensions, floating
dtype and finite pixels one slice at a time, then opens a zoomable browser. These
checks add an output-volume read and establish file consistency. Inspect fine
features and residuals before accepting the result.


In [ ]:
# 16. Denoise the full volume
VOLUME_CHECKPOINT = 'lcl'  # This cell has its own explicit checkpoint selection
volume_viewer = ct.denoise_volume(run, workspace, checkpoint=VOLUME_CHECKPOINT,
                                on_line=live_log_box())


## Optional queue inspection and repair: cells 17–18

Run cell 17 with its default empty selections to inspect the queue. The printed
`proposed.entries` contains each entry's `id` and `plan_path`; `preflight` reports
why an entry is valid, blocked or duplicated.

To remove one entry, copy its exact `id` into `REMOVE_ENTRY_IDS`, for example
`REMOVE_ENTRY_IDS = ['copied_entry_id']`, and rerun cell 17. To correct a moved
plan, put `{copied_entry_id: '/absolute/path/to/run_plan.json'}` into
`REPLACE_PLAN_PATHS`. Review the newly printed proposed entries, then set
`APPLY_QUEUE_REPAIR=True` in cell 18 and run it. Repair backs up the exact previous
queue bytes. Entry removal affects the queue only, not run files.

For corrupt queue JSON, set `REBUILD_FROM_PLANS` to an explicit list of valid
absolute plan paths in cell 17. Deduplication and removing confirmed missing
plans are separate opt-in settings; inspect their effects before applying.

If a lock is stale, first confirm its recorded job/kernel has stopped. Inspect
the lock's JSON, copy its token to `STALE_LOCK_TOKEN`, and use the confirmation
flags in cell 18. Never remove the lock of an active reconstruction.


In [ ]:
# 17. Preview queue repair (optional)
REMOVE_ENTRY_IDS = []
REPLACE_PLAN_PATHS = {}  # {stable_entry_id: absolute_plan_path}
DEDUPLICATE_QUEUE = False
REMOVE_CONFIRMED_MISSING = False
REBUILD_FROM_PLANS = None
queue_preview = queue.propose_queue_edit(workspace.queue_path, REMOVE_ENTRY_IDS,
    REPLACE_PLAN_PATHS, DEDUPLICATE_QUEUE, REMOVE_CONFIRMED_MISSING, REBUILD_FROM_PLANS)
print(json.dumps(queue_preview, indent=2))


In [ ]:
# 18. Apply queue repair (optional)
APPLY_QUEUE_REPAIR = False
REMOVE_STALE_QUEUE_LOCK = False
ORIGINAL_JOB_OR_KERNEL_STOPPED = False
STALE_LOCK_TOKEN = None  # Copy from the inspected queue lock
if REMOVE_STALE_QUEUE_LOCK:
    queue.remove_confirmed_stale_lock(queue.queue_lock(workspace.queue_path),
        ORIGINAL_JOB_OR_KERNEL_STOPPED, STALE_LOCK_TOKEN)
if APPLY_QUEUE_REPAIR:
    print('Backup:', queue.apply_queue_edit(workspace.queue_path, queue_preview))


## Optional scratch cleanup: cells 19–20

Use after stopping jobs that use the selected run. Cell 19 is a preview: select
categories in `CLEANUP_CATEGORIES`, rerun it, and inspect every proposed path,
file count, estimated bytes and stated effect. An empty list selects nothing.

| Category | Effect to consider |
|---|---|
| `full`, `even`, `odd` | Removes the corresponding reconstruction stack and invalidates its completion record; queue reconstruction can rebuild it |
| `projection_views` | Removes generated HDF5 view files, which point at the raw scan |
| `denoised` | Removes generated denoising outputs |
| `cor` | Selects generated COR review/preview paths listed in the preview |
| `logs` | Removes run logs; history cannot be regenerated from a deleted log |
| `training` | Removes model/checkpoint outputs and the scan training YAML, including saved normalization; resume/inference then require restoration or fresh training |

For deletion, copy the **exact resolved run directory** from the preview into
`CLEANUP_CONFIRM_RUN` in cell 20, set `CLEANUP_EXECUTE=True` and
`CLEANUP_DRY_RUN=False`, then run that cell. Defaults only preview. Rerun cell 19
after changing any selection. Raw scans, source code, environments, active locks
and symlink/junction paths are protected. Plans and queue entries are retained.

Preserve accepted raw data, TIFFs, model, YAML and plan in durable
storage before choosing cleanup. The report records failures and estimated
reclaimed bytes; it is not a storage backup and free-space updates may lag.


In [ ]:
# 19. Preview scratch cleanup (optional)
CLEANUP_CATEGORIES = []  # cor, projection_views, full, even, odd, denoised, logs, training
run.assert_current(run.snapshot(), check_source=True)
cleanup_plan = ct.read_plan(run.plan_path)
protected_paths = workspace.protected_paths(run)
cleanup_preview = cleanup.cleanup_manifest(cleanup_plan, workspace.output_root,
    CLEANUP_CATEGORIES, protected_paths=protected_paths)
print(json.dumps(cleanup_preview, indent=2))


In [ ]:
# 20. Delete reviewed scratch outputs (optional)
CLEANUP_DRY_RUN = True
CLEANUP_EXECUTE = False
CLEANUP_CONFIRM_RUN = ''  # Exact resolved run directory from the preview
cleanup_report = cleanup.execute_cleanup(cleanup_plan, workspace.output_root, cleanup_preview,
    CLEANUP_DRY_RUN, CLEANUP_CONFIRM_RUN, CLEANUP_EXECUTE, protected_paths=protected_paths)
print('Cleanup:', cleanup_report)
if cleanup_report['status'] != 'dry-run':
    run.invalidate()  # Inspect the run again after deleting generated outputs


## Decisions and recovery

| Situation | What to rerun or inspect |
|---|---|
| Kernel restarted | Run 1–4 with the original scan/run settings, then 6 to restore the plan; run 11–12 to reconnect stacks and training configuration |
| Stale viewer or replaced run | Rerun scan inspection/settings and recreate that viewer |
| COR review required | Choose informative rows/native bounds in 6, or review/apply an offset in 7 and save it in 8; run 5 if an alignment estimate is needed |
| Source/backend/settings differ | Restore the original inputs/settings or use a new run label; do not edit the frozen plan |
| Reconstruction interrupted | Inspect logs/report and rerun 9–10; completed stacks are validated and reused |
| Training interrupted | Preserve YAML/models/logs, run 11–12 with the original model settings, then enable 13 to resume |
| TrainOutput has no resume checkpoint | Preserve it; restore its checkpoint or use a new run label for fresh training |
| Volume inference interrupted | Preserve partial output; there is no automatic volume resume. Resolve the error before retrying with another destination/run |
| Queue needs repair | Preview in 17, apply explicitly in 18 |
| Generated outputs cleaned up | Rerun 3–4 and 6, then 9–12 as needed to rebuild/reconnect outputs |

Read the full log path printed by a failing action. The live box shows only a
tail. GPU utilization alone does not prove an epoch is stalled; check processes,
host RAM and allocation time in the compute terminal. Never remove active locks.

Compare several rows at matched zoom/contrast and inspect residuals for erased
features. Vo row agreement, low loss and a smooth image do not establish accuracy.


## Version and verification

v3.4 moves workflow implementations into `workflow/src/krg_workflow` and uses
explicit workspace and run objects in action cells. Source fingerprints, frozen
plans, locks and resume checks are retained. The architecture, training losses,
learning schedule and raw checkpoint criteria are unchanged. Early stopping monitors evaluation
total loss with a post-warmup moving average and minimum improvement threshold.
Display smoothing affects plots only.

Local checks cover imports, Python 3.8 syntax, saved plan/queue formats, command
generation, history plots, temporary TIFF/HDF5 data and inference. They do not
verify a fresh NERSC installation, ALS reconstruction or model training. See
the [workflow guide](../docs/WORKFLOW.md) for module boundaries and checks.
